# EDA 
This notebook explores the data evailable transit data for further development


## Packages

In [ ]:
import os 
from google.transit import gtfs_realtime_pb2
import requests
from requests.exceptions import Timeout, ConnectionError, HTTPError
from time import sleep
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from typing import Optional, List
import mysql.connector
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Constants

In [ ]:
realtime_vehicle_updates_url = "https://bct.tmix.se/gtfs-realtime/vehicleupdates.pb?operatorIds=47"

root_dir = os.path.dirname(os.path.abspath("."))  # Fixed typo from dirname to dir
data_dir = os.path.join(root_dir, "data")
src_dir = os.path.join(root_dir, "src")
os.makedirs(data_dir, exist_ok=True)

df_path = os.path.join(data_dir, "vehicle_updates.csv")
df_bus_stop_path = os.path.join(data_dir, "Kelowna_Regional_Transit_System_stops.csv")

## Fetch data 

In [ ]:


conn = mysql.connector.connect(
    host="localhost",
    user="transit",
    password="transit123",
    database="transitdb",
    port=3307
)

df_fetch = pd.read_sql("SELECT * FROM TRANSIT_DATA", conn)

conn.close()

df_bus_stop = pd.read_csv(df_bus_stop_path)
rename_cols = df_bus_stop.columns.tolist()[1:]
df_bus_stop.rename(columns=
            {col: col+"_stop" for col in rename_cols}, inplace=True)

In [ ]:


# Replace with your .shp file path
gdf = gpd.read_file(os.path.join(data_dir, "routes","routes.shp"))
gdf_8 = gdf[gdf['route_id'] == '8-KEL']

In [ ]:
df_8 = df_fetch[df_fetch['trip_route_id'] == '8-KEL']
df_8['stop_id'] = df_8['stop_id'].astype(int)
df_8 = pd.merge(df_8, df_bus_stop, left_on='stop_id', right_on='stopid', how='left', suffixes=(None, '_stop'))

In [ ]:
df_8_veh = df_8[df_8['vehicle_id'] == "3131329323"]
df_8['vehicle_id'].value_counts()

## Plots

### Stops

In [ ]:
df_stops = df_8.groupby('stop_id').first().reset_index()

In [ ]:
df_8_dir_0 = df_8[df_8['trip_direction_id'] == 0]
df_8_dir_1 = df_8[df_8['trip_direction_id'] == 1]


In [ ]:
df_dir_1_stop_seq = df_8_dir_1.groupby('current_stop_sequence')['stop_id'].value_counts().reset_index()
df_dir_0_stop_seq = df_8_dir_0.groupby('current_stop_sequence')['stop_id'].value_counts().reset_index()

In [ ]:
# check if stop id in df_dir_1_stop_seq are in df_dir_0_stop_seq
set_1 = set(df_dir_1_stop_seq['stop_id'].unique())
set_0 = set(df_dir_0_stop_seq['stop_id'].unique())
set_1 & set_0


In [ ]:
bool_unique_stop_per_sequence= df_dir_1_stop_seq['stop_id'].nunique() == len(df_dir_1_stop_seq)
print(f"Unique stop per sequence in direction 1: {bool_unique_stop_per_sequence}")
bool_unique_stop_per_sequence= df_dir_0_stop_seq['stop_id'].nunique() == len(df_dir_0_stop_seq)
print(f"Unique stop per sequence in direction 0: {bool_unique_stop_per_sequence}")

In [ ]:
df_stops_dir_0 = df_stops[df_stops['trip_direction_id'] == 0]
df_stops_dir_1 = df_stops[df_stops['trip_direction_id'] == 1]

In [ ]:

fig = go.Figure()

for i in range(len(gdf_8)):
    long, lat = gdf_8.iloc[i]['geometry'].xy
    long = list(long)
    lat = list(lat)
    fig.add_trace(go.Scattermap(
        lon=long,
        lat=lat,
        mode='lines',
        name=f'Segment {i+1}'
    ))

fig.add_trace(go.Scattermap(
    lon=df_stops_dir_0['longitude_stop'],
    lat=df_stops_dir_0['latitude_stop'],
    mode='markers',
    marker=go.scattermap.Marker(
        size=9,
        color='red'
    ),
    hovertext=df_stops_dir_0['current_stop_sequence'],
    name='Dir 0 Stop Locations'
))

fig.add_trace(go.Scattermap(
    lon=df_stops_dir_1['longitude_stop'],
    lat=df_stops_dir_1['latitude_stop'],
    mode='markers',
    marker=go.scattermap.Marker(
        size=9,
        color='blue'
    ),
    hovertext=df_stops_dir_1['current_stop_sequence'],
    name='Dir 1 Stop Locations'
))

fig.update_layout(
    map_style="open-street-map",
    map=dict(
        center=dict(lat=df_stops_dir_1['latitude_stop'].mean(), 
                    lon=df_stops_dir_1['longitude_stop'].mean()),
        zoom=11
    ),
    showlegend=True
)

fig.show()

### Time series

In [188]:

ts = df_8_veh.sort_values('read_timestamp')
# Use existing `ts` sorted by read_timestamp
fig_ts = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=["Position Latitude", "Position Longitude", "Current Stop Sequence", "Trip Direction ID"]
)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['position_latitude'],
    mode='lines+markers', name='Position Latitude'
), row=1, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['position_longitude'],
    mode='lines+markers', name='Position Longitude'
), row=2, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['current_stop_sequence'],
    mode='lines+markers', name='Current Stop Sequence'
), row=3, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['trip_direction_id'],
    mode='lines+markers', name='Trip Direction ID'
), row=4, col=1)

fig_ts.update_yaxes(title_text='Position Latitude', row=1, col=1)
fig_ts.update_yaxes(title_text='Position Longitude', row=2, col=1)
fig_ts.update_yaxes(title_text='Current Stop Sequence', row=3, col=1)
fig_ts.update_yaxes(title_text='Trip Direction ID', row=4, col=1)
fig_ts.update_xaxes(title_text='Read Timestamp', row=4, col=1)

fig_ts.update_layout(height=900, showlegend=False)
fig_ts.show()


In [190]:
start_time = "2025-12-27 22:00:00"
end_time = "2025-12-27 22:35:00"

df_8_veh_time_filtered = df_8_veh[(df_8_veh['read_timestamp'] >= start_time) & (df_8_veh['read_timestamp'] <= end_time)]

In [191]:

ts = df_8_veh_time_filtered.sort_values('read_timestamp')
# Use existing `ts` sorted by read_timestamp
fig_ts = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=["Position Latitude", "Position Longitude", "Current Stop Sequence", "Trip Direction ID"]
)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['position_latitude'],
    mode='lines+markers', name='Position Latitude'
), row=1, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['position_longitude'],
    mode='lines+markers', name='Position Longitude'
), row=2, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['current_stop_sequence'],
    mode='lines+markers', name='Current Stop Sequence'
), row=3, col=1)

fig_ts.add_trace(go.Scatter(
    x=ts['read_timestamp'], y=ts['trip_direction_id'],
    mode='lines+markers', name='Trip Direction ID'
), row=4, col=1)

fig_ts.update_yaxes(title_text='Position Latitude', row=1, col=1)
fig_ts.update_yaxes(title_text='Position Longitude', row=2, col=1)
fig_ts.update_yaxes(title_text='Current Stop Sequence', row=3, col=1)
fig_ts.update_yaxes(title_text='Trip Direction ID', row=4, col=1)
fig_ts.update_xaxes(title_text='Read Timestamp', row=4, col=1)

fig_ts.update_layout(height=900, showlegend=False)
fig_ts.show()
